In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import invgamma, invwishart, gaussiankde
from cavi import run_cavi
from gibbs import run_gibbs
from data_prep import prep_data

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
# ── Data generation ──────────────────────────────────────────────────────────

T, C, N, L, W = 100, 3, 3, 2, 1
K = N*L + W
T_sim = T + L

def companion(B, N, L):
    K = N * L
    C = np.zeros((K, K))
    C[:N, :] = B[:, :K]  # exclude w column
    C[N:, :-N] = np.eye(N * (L - 1))
    return C

betas = np.random.normal(0.2, 0.3, size=(C, N, K))
for c in range(C):
    comp = companion(betas[c], N, L)
    radius = np.max(np.abs(np.linalg.eigvals(comp)))
    if radius >= 1:
        betas[c] *= 0.8 / radius

gamma = np.zeros((C, N))
for c in range(C):
    gamma[c] = np.random.normal(c * 0.2 * (-1)**c, 0.1 + c * 0.05, size=N)

# innovations
innovations = np.zeros((T_sim, C, N))
for c in range(C):
    A = np.random.randn(N, N) * 0.1
    Sigma = A @ A.T + np.eye(N) * 0.01
    innovations[:, c, :] = np.random.multivariate_normal(np.zeros(N), Sigma, size=T_sim)

# exogenous variables
z = np.zeros(T_sim)
for t in range(1, T_sim):
    z[t] = 0.7*z[t-1] + np.random.randn()

w = np.zeros(T_sim)
for t in range(1, T_sim):
    w[t] = 0.3*w[t-1] + np.random.randn()

# simulate
Y = np.random.randn(T_sim, C, N)
for t in range(L, T_sim):
    for c in range(C):
        lags = np.concatenate([Y[t-1, c, :], Y[t-2, c, :], [w[t-1]]])
        Y[t, c, :] = betas[c] @ lags + gamma[c]*z[t-1] + innovations[t, c, :]


In [20]:
cavi_pack, gibbs_pack = prep_data(Y, w, z, C, N, T, K, L)

In [29]:
results_cavi, ELBO = run_cavi(cavi_pack, C, N, K, T)

In [5]:
results_gibbs, ess, rhat = run_gibbs(gibbs_pack, C, N, K, T)

In [ ]:
cov_true = []
for c in range(C):
    delta_c_draws = np.array([results_gibbs["delta_c"][t][c] for t in range(len(results_gibbs["delta_c"]))])
    V_deltac_true = np.cov(delta_c_draws.T)
    cov_true.append(V_deltac_true)

idx_deltac = cavi_pack["idx_deltac"]
size_deltac = cavi_pack["size_deltac"]

V_delta = results_cavi["V_delta"]
cov_cavi = []
for c in range(C):
    start = idx_deltac[c]
    V_deltac_cavi = V_delta[start:start + size_deltac, start:start + size_deltac]
    cov_cavi.append(V_deltac_cavi)

def UQF(cov_true, cov_est):
    inv_cov_est = np.linalg.inv(cov_est)
    joint = cov_true @ inv_cov_est
    max_eigenvalue = np.max(np.linalg.eigvalsh(joint))

    return 1/max_eigenvalue

UQFs_cavi = [UQF(cov_true[c], cov_cavi[c]) for c in range(C)]

UQFs_cavi

[np.float64(0.0065812837036667415),
 np.float64(0.002635037944713193),
 np.float64(0.0014036890120357063)]

In [ ]:
# CAVI samples
mu_delta, V_delta, v_bar, s_bar, S_bar_sigma = results_cavi.values()

cavi_samples = {'beta_0': [], 'lam': [], 'beta_c': [], 'gamma_c': [], 'delta_c': [], 'Sigma_c': []}

n_samples = len(results_gibbs["beta_c"])

for n in range(n_samples):
    delta =  np.random.multivariate_normal(mu_delta, V_delta)
    beta_0 = delta[:idx_deltac[0]]
    cavi_samples['beta_0'].append(beta_0.copy())

    beta_c = [delta[idx_deltac[c]:idx_deltac[c] + N*K] for c in range(C)]
    cavi_samples['beta_c'].append(beta_c.copy())
    gamma_c = [delta[idx_deltac[c] + N*K:idx_deltac[c] + N*K + size_gammac] for c in range(C)]
    cavi_samples['gamma_c'].append(gamma_c.copy())
    delta_c = [delta[idx_deltac[c]:idx_deltac[c] + size_deltac] for c in range(C)]
    cavi_samples['delta_c'].append(delta_c.copy())

    lam = invgamma.rvs(s_bar/2, scale=v_bar/2)
    cavi_samples['lam'].append(lam)

    Sigma_c = [invwishart.rvs(T, S_bar_sigma[c]) for c in range(C)]
    cavi_samples['Sigma_c'].append(Sigma_c.copy())

In [ ]:
def faes_accuracy(vi_samples, gibbs_samples, positive_support=False):
    if positive_support:
        vi_samples = np.log(vi_samples)
        gibbs_samples = np.log(gibbs_samples)
    
    # Fit KDEs
    kde_q = gaussian_kde(vi_samples)
    kde_p = gaussian_kde(gibbs_samples)
    
    # Common grid
    lo = min(vi_samples.min(), gibbs_samples.min()) - 3*np.std(gibbs_samples)
    hi = max(vi_samples.max(), gibbs_samples.max()) + 3*np.std(gibbs_samples)
    grid = np.linspace(lo, hi, 2000)
    
    iae = np.trapz(np.abs(kde_q(grid) - kde_p(grid)), grid)
    return 100 * (1 - 0.5 * iae)